# Demand Forecasting Model using PySpark

## Project Description
This project aims to predict e-commerce product demand using PySpark to improve supply chain efficiency, reduce stockouts, and optimize inventory management. The model leverages historical sales data and advanced analytics to generate accurate demand forecasts.

## Objectives
* Predict future product demand
* Identify seasonal trends and purchasing patterns
* Optimize inventory and supply chain decisions
* Improve business efficiency and reduce costs

## 1. Data Engineer (Ingestion & Storage)

**Responsibilities:**
* Read data
* Perform data cleaning and preprocessing using PySpark
* Store data in distributed systems (HDFS / Data Lake)

### Initialize Spark Session and Generate Synthetic Data

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, lit, rand, monotonically_increasing_id
import pandas as pd
import numpy as np
from pyspark.sql.types import StructType, StructField, DateType, StringType, DoubleType, IntegerType

# Docker/Hadoop environment settings (matches hadoop.env and docker-compose.yml)
HDFS_URI = os.getenv("CORE_CONF_fs_defaultFS", "hdfs://namenode:9000")
YARN_RM = os.getenv("YARN_CONF_yarn_resourcemanager_hostname", "resourcemanager")

import os

# Set a dummy HADOOP_CONF_DIR to satisfy Spark's strict startup check
dummy_conf_dir = "/tmp/hadoop-conf"
os.makedirs(dummy_conf_dir, exist_ok=True)
os.environ["HADOOP_CONF_DIR"] = dummy_conf_dir
# Initialize Spark Session for Hadoop + YARN connectivity
spark = (
    SparkSession.builder
    .appName("DemandForecasting")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.hadoop.fs.defaultFS", HDFS_URI)
    .config("spark.hadoop.yarn.resourcemanager.hostname", YARN_RM)
    .config("spark.hadoop.dfs.replication", os.getenv("HDFS_CONF_dfs_replication", "1"))
    .getOrCreate()
)

print(f"Spark master: {spark.sparkContext.master}")
print(f"Spark app name: {spark.sparkContext.appName}")
print(f"HDFS URI: {HDFS_URI}")
print(f"YARN RM: {YARN_RM}")

# Generate synthetic data for demonstration
# In production, replace this with data loaded from HDFS/S3.
num_rows = 10000
start_date = pd.to_datetime('2022-01-01')
dates = pd.to_datetime([start_date + pd.Timedelta(days=np.random.randint(0, 365)) for _ in range(num_rows)])

products = [f'product_{i}' for i in range(10)]
stores = [f'store_{i}' for i in range(5)]

data = {
    'order_date': dates,
    'product_id': [np.random.choice(products) for _ in range(num_rows)],
    'store_id': [np.random.choice(stores) for _ in range(num_rows)],
    'quantity': np.random.randint(1, 100, num_rows),
    'price': np.round(np.random.uniform(5.0, 500.0, num_rows), 2),
    'is_promo': np.random.choice([0, 1], num_rows, p=[0.8, 0.2])
}

pd_df = pd.DataFrame(data)

# Introduce some missing values for demonstration of cleaning
pd_df.loc[np.random.choice(pd_df.index, 50, replace=False), 'quantity'] = np.nan
pd_df.loc[np.random.choice(pd_df.index, 20, replace=False), 'price'] = np.nan

# Explicitly define schema to avoid inference issues with numpy types
schema = StructType([
    StructField("order_date", DateType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("quantity", DoubleType(), True),  # Using DoubleType as quantity can be NaN (float)
    StructField("price", DoubleType(), True),
    StructField("is_promo", IntegerType(), True)
])

spark_df = spark.createDataFrame(pd_df, schema=schema)
spark_df.printSchema()
spark_df.show(5)

26/04/24 21:11:19 WARN Utils: Your hostname, abdo-HP-ZBook-17-G3 resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface wlp2s0)
26/04/24 21:11:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 21:11:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Hadoop/HDFS connectivity smoke test
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
print("fs.defaultFS:", hadoop_conf.get("fs.defaultFS"))
print("yarn.resourcemanager.hostname:", hadoop_conf.get("yarn.resourcemanager.hostname"))

test_path = f"{HDFS_URI}/tmp/demand_forecasting_smoke_test"
spark.range(1).write.mode("overwrite").parquet(test_path)
print(f"Wrote smoke test data to: {test_path}")

smoke_df = spark.read.parquet(test_path)
smoke_df.show()

### Data Cleaning and Preprocessing

In [ ]:
# Convert 'order_date' to date type and fill missing values
spark_df = spark_df.withColumn('order_date', to_date(col('order_date')))

# Fill missing 'quantity' with the mean, 'price' with median (example)
# For simplicity, we'll use a fixed value or mean from a collected statistic
# In a real scenario, you'd compute this dynamically or use more sophisticated imputation.
mean_quantity = spark_df.agg({'quantity': 'mean'}).collect()[0][0]
median_price = spark_df.approxQuantile('price', [0.5], 0.01)[0]

spark_df_cleaned = spark_df.fillna({'quantity': mean_quantity, 'price': median_price})

# Calculate total sales
spark_df_cleaned = spark_df_cleaned.withColumn('total_sales', col('quantity') * col('price'))

spark_df_cleaned.printSchema()
spark_df_cleaned.show(5)

print(f"Missing values in 'quantity' after imputation: {spark_df_cleaned.filter(col('quantity').isNull()).count()}")
print(f"Missing values in 'price' after imputation: {spark_df_cleaned.filter(col('price').isNull()).count()}")

In [ ]:
# Persist cleaned data to HDFS
cleaned_data_hdfs_path = f"{HDFS_URI}/user/data-engineer/demand_forecasting/cleaned_data"
spark_df_cleaned.write.mode("overwrite").parquet(cleaned_data_hdfs_path)
print(f"Saved cleaned data to: {cleaned_data_hdfs_path}")

# Read back from HDFS to verify
spark_df_cleaned_hdfs = spark.read.parquet(cleaned_data_hdfs_path)
print("Rows loaded back from HDFS:", spark_df_cleaned_hdfs.count())
spark_df_cleaned_hdfs.show(5)

## 2. Data Analyst (Exploration & Insights)

**Responsibilities:**
* Perform Exploratory Data Analysis (EDA)
* Analyze trends and patterns in sales data
* Generate insights and visualizations

### Basic Statistics and Schema

In [ ]:
spark_df_cleaned.describe().show()

# Convert to Pandas for easier visualization (for small datasets)
# For large datasets, use PySpark's DataFrame operations or distributed visualization tools
pd_df_for_eda = spark_df_cleaned.toPandas()

### Sales Trends Over Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate daily sales
daily_sales = pd_df_for_eda.groupby('order_date')['total_sales'].sum().reset_index()
daily_sales = daily_sales.sort_values('order_date')

plt.figure(figsize=(12, 6))
sns.lineplot(x='order_date', y='total_sales', data=daily_sales)
plt.title('Daily Total Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.grid(True)
plt.show()

### Sales by Product and Store

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Sales by Product
product_sales = pd_df_for_eda.groupby('product_id')['total_sales'].sum().sort_values(ascending=False).reset_index()
sns.barplot(x='product_id', y='total_sales', data=product_sales, ax=axes[0])
axes[0].set_title('Total Sales by Product')
axes[0].set_xlabel('Product ID')
axes[0].set_ylabel('Total Sales')
axes[0].tick_params(axis='x', rotation=45)

# Sales by Store
store_sales = pd_df_for_eda.groupby('store_id')['total_sales'].sum().sort_values(ascending=False).reset_index()
sns.barplot(x='store_id', y='total_sales', data=store_sales, ax=axes[1])
axes[1].set_title('Total Sales by Store')
axes[1].set_xlabel('Store ID')
axes[1].set_ylabel('Total Sales')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Machine Learning Engineer (Model Development)

**Responsibilities:**
* Build and train forecasting models
* Perform feature engineering (lag features, seasonal features)
* Evaluate model performance

**Models Used (Conceptual):**
* Linear Regression
* Random Forest
* Gradient Boosting
* Time Series Models (ARIMA / Prophet)

### Feature Engineering

In [ ]:
from pyspark.sql.functions import dayofmonth, dayofweek, month, year, weekofyear, lag, monotonically_increasing_id
from pyspark.sql.window import Window

# Aggregate data to daily level for forecasting
daily_agg_df = spark_df_cleaned.groupBy('order_date', 'product_id', 'store_id') \
                               .agg({'quantity': 'sum', 'total_sales': 'sum'}) \
                               .withColumnRenamed('sum(quantity)', 'daily_quantity') \
                               .withColumnRenamed('sum(total_sales)', 'daily_total_sales')

# Sort by date, product, and store for lag features
window_spec = Window.partitionBy('product_id', 'store_id').orderBy('order_date')

# Add time-based features
daily_agg_df = daily_agg_df.withColumn('day_of_month', dayofmonth(col('order_date'))) \
                           .withColumn('day_of_week', dayofweek(col('order_date'))) \
                           .withColumn('month', month(col('order_date'))) \
                           .withColumn('year', year(col('order_date'))) \
                           .withColumn('week_of_year', weekofyear(col('order_date')))

# Add lag features (e.g., previous day's quantity, previous week's quantity)
daily_agg_df = daily_agg_df.withColumn('lag_1_day_qty', lag(col('daily_quantity'), 1).over(window_spec))
# daily_agg_df = daily_agg_df.withColumn('lag_7_day_qty', lag(col('daily_quantity'), 7).over(window_spec)) # Uncomment for more lag features

# Fill NaN created by lag features (e.g., with 0 or mean, depending on domain)
daily_agg_df = daily_agg_df.fillna(0, subset=['lag_1_day_qty'])

daily_agg_df.show(5)
daily_agg_df.printSchema()

### Prepare Data for ML Model

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

# Convert categorical features to numerical using StringIndexer and OneHotEncoder
indexer_product = StringIndexer(inputCol="product_id", outputCol="product_idx")
indexer_store = StringIndexer(inputCol="store_id", outputCol="store_idx")

encoder_product = OneHotEncoder(inputCol="product_idx", outputCol="product_vec")
encoder_store = OneHotEncoder(inputCol="store_idx", outputCol="store_vec")

# Define features and label
feature_cols = ['day_of_month', 'day_of_week', 'month', 'year', 'week_of_year', 'lag_1_day_qty', 'product_vec', 'store_vec']
label_col = 'daily_quantity'

# Assemble features into a single vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Create a pipeline to apply transformations
pipeline = Pipeline(stages=[
    indexer_product,
    indexer_store,
    encoder_product,
    encoder_store,
    assembler
])

# Fit and transform the data
ml_df = pipeline.fit(daily_agg_df).transform(daily_agg_df)

# Select final features and label
model_data = ml_df.select("features", label_col)
model_data.show(5)


### Split Data (Train/Test)

In [ ]:
# Split data into training and testing sets
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training Data Count: {train_data.count()}")
print(f"Testing Data Count: {test_data.count()}")

### Build and Train Linear Regression Model

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol=label_col, maxIter=10, regParam=0.3, elasticNetParam=0.8)

# Train the model
lr_model = lr.fit(train_data)

# Print the coefficients and intercept for linear regression
print("Coefficients: " + str(lr_model.coefficients))
print("Intercept: " + str(lr_model.intercept))

# Summarize the model over the training set
trainingSummary = lr_model.summary
print(f"R Squared (training): {trainingSummary.r2}")

### Evaluate Model Performance

In [ ]:
# Make predictions on test data
predictions = lr_model.transform(test_data)

# Show predictions
predictions.select("features", label_col, "prediction").show(5)

# Evaluate the model
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")
print(f"R Squared (R2) on test data = {r2}")

## 4. Big Data Engineer (Optimization & Performance)

**Responsibilities:**
* Optimize Spark jobs for performance
* Manage partitioning and distributed processing

### Replication (Conceptual)
* Ensures fault tolerance by storing multiple copies of data.
* Typically handled in HDFS with a replication factor (e.g., 3).
* In a cloud environment, this is managed by the underlying storage service (e.g., GCS, S3).

### Repartition (Conceptual)
* Redistributes data across partitions for better parallel processing.
* Used before joins and aggregations to prevent data skew and improve performance.
* Example usage:
  ```python
  # Repartition DataFrame to 100 partitions
  optimized_df = spark_df_cleaned.repartition(100)
  
  # Repartition by a specific column for better locality before joins/aggregations
  optimized_df = spark_df_cleaned.repartition(col("product_id"))
  ```

### Caching (Conceptual)
* Persists DataFrame or RDD in memory or on disk for faster access in iterative algorithms or multiple actions.
* Example usage:
  ```python
  # Cache the DataFrame for repeated access
  model_data.cache()
  # Perform operations
  # model_data.unpersist() # To uncache when no longer needed
  ```

## 5. MLOps Engineer (Deployment & Monitoring)

**Responsibilities:**
* Deploy the model into production
* Automate data pipelines
* Monitor model performance and accuracy

### Model Deployment (Conceptual)
In a real-world scenario, the trained `lr_model` would be saved and deployed:

1.  **Save the Model:**
    ```python
    # lr_model.write().overwrite().save("/path/to/save/demand_forecasting_model")
    ```
2.  **Load the Model for Inference:**
    ```python
    # from pyspark.ml.regression import LinearRegressionModel
    # loaded_model = LinearRegressionModel.load("/path/to/save/demand_forecasting_model")
    # predictions = loaded_model.transform(new_inference_data)
    ```
3.  **Deployment Environment:** Deploy on platforms like Kubernetes (with Spark on Kubernetes), Databricks, Amazon SageMaker, Google AI Platform, or Azure Machine Learning.

### Data and Model Pipelines (Conceptual)
*   **Automated Data Ingestion:** Scheduled jobs (e.g., Airflow, Google Cloud Composer, Azure Data Factory) to pull new sales data.
*   **Automated Feature Engineering:** New data goes through the same feature engineering pipeline (`Pipeline` object).
*   **Automated Retraining:** Periodically retrain the model with fresh data to maintain accuracy.

### Monitoring (Conceptual)
*   **Model Performance Monitoring:** Track RMSE, R2, MAE, etc., on new data. Detect model drift (when model performance degrades over time).
*   **Data Quality Monitoring:** Ensure input data quality (no missing values, correct types, within expected ranges).
*   **System Health Monitoring:** Monitor Spark job success/failure rates, resource utilization, latency.
*   **Alerting:** Set up alerts for significant drops in performance or data anomalies.

## Deliverables
*   Cleaned and processed dataset (`spark_df_cleaned`)
*   Demand forecasting model (Conceptualized with `lr_model`)
*   Optimized Spark pipeline (Conceptualized with `Pipeline` and `repartition` explanation)
*   Visualization (EDA plots)
*   Full project documentation (This notebook)

In [ ]:
# Stop Spark Session
spark.stop()